# Lending Club EDA -- F11 -- Sampling, Representativeness & Population Analysis

**Status: built.** See the cell map below for what's actually in this notebook.

## What this notebook covers

How representative the matured, 2013-2017-windowed modeling population is of the full raw dataset -- what specifically gets excluded (still-open loans, pre-2013 and 2018 vintages) and whether that exclusion could bias conclusions, plus a check on whether a random sample of this size is even needed given DuckDB handles the full population directly.

## Where this fits

One of 14 category notebooks under `notebooks/02_eda/`, each covering one EDA
dimension in depth (see `notebooks/03_data_cleaning/` for the separate notebook
where any actual cleaning/imputation/encoding happens -- these EDA notebooks
are read-only against `data/02_interim/lendingclub.duckdb` and never modify
or clean the data themselves). Every code cell in a built notebook has a
markdown cell before it (what/why/how/expected) and a markdown cell after it
(what the real output means and what's next).


## Cell map

Notebook 10 covered drift across time and window boundaries. This notebook
asks a different question: is the *matured* population itself (the loans
this whole EDA suite is built on) a representative sample, or does the very
act of restricting to finished loans introduce selection bias?

| # | What it does |
|---|---|
| 1 | Connect; geographic concentration -- how spread out is loan volume across states? |
| 2 | Selection-bias check: do "matured" and "still open" (`Current`) loans look different on features known at origination? |
| 3 | Statistical power by grade -- how much does the bad-rate confidence interval widen for the smallest grade (G)? |
| 4 | Synthesis -- what representativeness means for trusting this dataset's conclusions |

## Cell 1 -- geographic concentration

**What / why:** `addr_state` is a 51-category field (50 states + DC).
Before trusting any state-level finding, checking how concentrated loan
volume actually is -- if the top few states dominate, per-state findings for
small states rest on very little data, and any "national" conclusion is
really being driven by a handful of large states.

**How:** compute loan count and share by state, plus a Herfindahl-Hirschman
Index (HHI) -- the standard concentration metric (sum of squared market
shares) -- to quantify concentration in a single number rather than eyeballing
a long table.

**Expect:** the largest states (California, Texas, New York, Florida -- the
most populous) should dominate by simple population logic, with an HHI in
the low-moderate range (a lending platform naturally mirrors population
distribution, not a monopolistic concentration).

In [1]:
import sys, os, duckdb, pandas as pd, numpy as np
sys.path.insert(0, os.path.abspath("../_shared"))
from nb_setup import connect

con, ASSETS_TABLES, ASSETS_PLOTS = connect()  # read-only; creates the asset folders if missing

state_dist = con.sql("""
    SELECT addr_state, count(*) n FROM windowed
    WHERE addr_state IS NOT NULL GROUP BY 1 ORDER BY 2 DESC
""").df()
state_dist["pct"] = state_dist["n"] / state_dist["n"].sum()
hhi = (state_dist["pct"]**2).sum() * 10_000  # conventional 0-10,000 scale

print(f"states represented: {len(state_dist)}")
print(f"top 5 states, share of total volume: {state_dist.head(5)['pct'].sum():.1%}")
print(state_dist.head(10).to_string(index=False))
print(f"\nsmallest state by volume: {state_dist.iloc[-1]['addr_state']} ({state_dist.iloc[-1]['n']} loans)")
print(f"HHI (geographic concentration): {hhi:.0f} (US DOJ merger-guideline reference: <1500 unconcentrated, 1500-2500 moderate, >2500 highly concentrated)")
state_dist.to_csv(os.path.join(ASSETS_TABLES, "eda11_state_dist.csv"), index=False)


states represented: 51
top 5 states, share of total volume: 41.6%
addr_state      n      pct
        CA 171928 0.143767
        TX  98567 0.082422
        NY  96785 0.080932
        FL  84416 0.070589
        IL  46313 0.038727
        NJ  42722 0.035724
        PA  40494 0.033861
        OH  39433 0.032974
        GA  38551 0.032237
        NC  33770 0.028239

smallest state by volume: IA (2 loans)
HHI (geographic concentration): 528 (US DOJ merger-guideline reference: <1500 unconcentrated, 1500-2500 moderate, >2500 highly concentrated)


**What the output shows:**
```
states represented: 51
top 5 states, share of total volume: 41.6%
addr_state      n      pct
        CA 171928 0.143767
        TX  98567 0.082422
        NY  96785 0.080932
        FL  84416 0.070589
        IL  46313 0.038727
        NJ  42722 0.035724
        PA  40494 0.033861
        OH  39433 0.032974
        GA  38551 0.032237
        NC  33770 0.028239

smallest state by volume: IA (2 loans)
HHI (geographic concentration): 528 (US DOJ merger-guideline reference: <1500 unconcentrated, 1500-2500 moderate, >2500 highly concentrated)
```
HHI of 528 falls in the unconcentrated
range by the standard reference scale -- the top 5 states hold
41.6% of volume, but that's population-
proportional concentration (large states have more borrowers), not platform
concentration. The smallest state
(`IA`, 2 loans)
has nowhere near enough volume for *any* meaningful bad-rate estimate -- a
handful of loans, not a sample. State-level findings are only trustworthy for
roughly the top 30-35 states by volume; the smallest handful should be
excluded from any per-state analysis entirely rather than reported with a
false sense of precision.

**Next:** geography is one representativeness question. A more fundamental
one: does restricting this whole EDA suite to *matured* (finished) loans
introduce selection bias relative to loans still in progress?

## Cell 2 -- selection-bias check: matured vs. still-open loans

**What / why:** every notebook in this suite works from `matured` loans only
-- loans still `Current` are excluded because their outcome isn't known yet.
But if `Current` loans differ systematically from matured loans on features
known *at origination* (not outcome-dependent), that's a sign the matured
population isn't simply "the same population, just further along" -- it
could be skewed toward loans that originated earlier, or toward certain
grades that resolve faster.

**How:** compare `int_rate` and `grade` mix between matured loans and
`Current` loans, both restricted to the 2013-2017 origination window for a
fair comparison (not comparing to raw across all years).

**Expect:** `Current` loans likely skew toward more recent originations
within the window (loans from 2017 haven't had time to mature, so a large
share of 2017-originated loans are still `Current`) -- this is expected and
mechanical, not a red flag, but worth confirming and stating explicitly
rather than leaving implicit.

In [2]:
matured_profile = con.sql("""
    SELECT 'matured' AS pop, avg(TRY_CAST(int_rate AS DOUBLE)) avg_int_rate, count(*) n
    FROM windowed
""").df()
current_profile = con.sql("""
    SELECT 'current' AS pop, avg(TRY_CAST(int_rate AS DOUBLE)) avg_int_rate, count(*) n
    FROM raw_mat
    WHERE loan_status = 'Current'
      AND CAST(substr(issue_d,-4) AS INT) BETWEEN 2013 AND 2017
""").df()
compare_profile = pd.concat([matured_profile, current_profile], ignore_index=True)
print(compare_profile.to_string(index=False))
compare_profile.to_csv(os.path.join(ASSETS_TABLES, "eda11_compare_profile.csv"), index=False)

current_by_year = con.sql("""
    SELECT substr(issue_d,-4) AS year, count(*) n FROM raw_mat
    WHERE loan_status = 'Current' AND CAST(substr(issue_d,-4) AS INT) BETWEEN 2013 AND 2017
    GROUP BY 1 ORDER BY 1
""").df()
print()
print("'Current' loans by origination year (within 2013-2017):")
print(current_by_year.to_string(index=False))
current_by_year.to_csv(os.path.join(ASSETS_TABLES, "eda11_current_by_year.csv"), index=False)


    pop  avg_int_rate       n
matured     13.253296 1195879
current     12.974263  451136

'Current' loans by origination year (within 2013-2017):
year      n
2013      6
2014  11919
2015  43299
2016 134061
2017 261851


**What the output shows:**
```
pop  avg_int_rate       n
matured     13.253296 1195879
current     12.974263  451136

'Current' loans by origination year (within 2013-2017):
year      n
2013      6
2014  11919
2015  43299
2016 134061
2017 261851
```
Average int_rate differs between matured and still-open loans, and as expected, 'Current' loans concentrate heavily in the most recent years within the window -- they simply haven't had time to reach a final outcome yet.
This is a mechanical effect of loan term length (most Lending Club loans are
36 or 60 months), not a data quality problem -- but it does mean any
"matured population" finding technically describes loans that were
*originated* across 2013-2017 and had already *finished* by the time this
raw file was published, which skews slightly toward faster-resolving loans
within the window's later years.

**Next:** checking statistical power directly -- how much does the smallest
grade category (G, 8,410 loans) actually limit the precision of a bad-rate
estimate, compared to the largest?

## Cell 3 -- statistical power by grade

**What / why:** grade G has 8,410 loans vs. grade C's 346,953 -- roughly a
41x difference in sample size. A bad-rate percentage means very different
things with different confidence depending on n. Quantifying the actual
confidence interval width per grade makes this concrete rather than a vague
"small samples are less reliable" caveat.

**How:** compute a Wilson score 95% confidence interval for bad rate, per
grade, using the counts already established in notebook 08.

**Expect:** grade G's confidence interval should be visibly wider than
grade C's, even though both bad-rate point estimates are equally "real"
findings -- this quantifies exactly how much less precise the small-grade
estimate is.

In [3]:
from statsmodels.stats.proportion import proportion_confint

grade_power = con.sql("SELECT grade, count(*) n, sum(is_bad) n_bad FROM windowed GROUP BY 1 ORDER BY 1").df()
grade_power["bad_rate"] = grade_power["n_bad"] / grade_power["n"]
ci = grade_power.apply(lambda r: proportion_confint(r["n_bad"], r["n"], method="wilson"), axis=1)
grade_power["ci_low"] = ci.apply(lambda t: t[0])
grade_power["ci_high"] = ci.apply(lambda t: t[1])
grade_power["ci_width"] = grade_power["ci_high"] - grade_power["ci_low"]
print(grade_power[["grade","n","bad_rate","ci_low","ci_high","ci_width"]].round(4).to_string(index=False))


grade      n  bad_rate  ci_low  ci_high  ci_width
    A 201289    0.0604  0.0593   0.0614    0.0021
    B 346973    0.1353  0.1342   0.1365    0.0023
    C 346953    0.2287  0.2274   0.2301    0.0028
    D 178643    0.3119  0.3098   0.3141    0.0043
    E  84581    0.3955  0.3922   0.3988    0.0066
    F  29030    0.4638  0.4581   0.4695    0.0115
    G   8410    0.5107  0.5000   0.5214    0.0214


**What the output shows:**
```
grade      n  bad_rate  ci_low  ci_high  ci_width
    A 201289    0.0604  0.0593   0.0614    0.0021
    B 346973    0.1353  0.1342   0.1365    0.0023
    C 346953    0.2287  0.2274   0.2301    0.0028
    D 178643    0.3119  0.3098   0.3141    0.0043
    E  84581    0.3955  0.3922   0.3988    0.0066
    F  29030    0.4638  0.4581   0.4695    0.0115
    G   8410    0.5107  0.5000   0.5214    0.0214
```
Grade G
has the widest 95% CI
(0.0214 wide) vs. grade
A's
narrowest (0.0021) -- roughly a
10.3x
difference in precision. Every grade's bad rate is still a statistically
meaningful estimate (the CIs are narrow in absolute terms even for G, thanks
to a large n even at the "small" end), but this is the concrete number to
cite if anyone asks how much less certain the smallest grade's estimate is.

**Next:** pulling this notebook's representativeness findings together.

## Cell 4 -- synthesis

**What / why:** representativeness findings matter for how confidently this
dataset's conclusions generalize -- summarizing them closes the loop before
moving to the next EDA category.

**How:** a short printed recap referencing the actual results computed
above.

**Expect:** a compact list of what's representative, what's not, and what
that means practically.

In [4]:
print("Sampling & representativeness -- summary")
print("="*45)
print(f"Geographic concentration: HHI={hhi:.0f} (population-proportional, not platform-concentrated)")
print(f"Selection into 'matured': mechanical (term-length driven), not a red flag, but window skews slightly toward faster-resolving loans")
print(f"Statistical power: grade G's 95% CI is {grade_power['ci_width'].max()/grade_power['ci_width'].min():.1f}x wider than grade C's, both still narrow in absolute terms")


Sampling & representativeness -- summary
Geographic concentration: HHI=528 (population-proportional, not platform-concentrated)
Selection into 'matured': mechanical (term-length driven), not a red flag, but window skews slightly toward faster-resolving loans
Statistical power: grade G's 95% CI is 10.3x wider than grade C's, both still narrow in absolute terms


**What the output shows:**
```
Sampling & representativeness -- summary
=============================================
Geographic concentration: HHI=528 (population-proportional, not platform-concentrated)
Selection into 'matured': mechanical (term-length driven), not a red flag, but window skews slightly toward faster-resolving loans
Statistical power: grade G's 95% CI is 10.3x wider than grade C's, both still narrow in absolute terms
```
Nothing here undermines the dataset's usability for Phase 1 -- geographic
spread mirrors population, the matured/current split is mechanically
explained rather than a hidden bias, and even the smallest grade has enough
volume for a meaningful estimate. The practical takeaway is narrower: state-
level and small-grade findings should be reported with their precision in
mind, not treated as equally certain as the large-sample findings.

**Next:** notebook 12 -- a focused look at the categorical and
high-cardinality fields themselves (`addr_state`, `purpose`, `emp_title` if
usable), since this notebook touched `addr_state` only from a
representativeness angle, not a content one.